# Model Quantisation — A Practical, Hands-On Guide

### Run Large Language Models *fast* on your own laptop (no expensive GPU needed)

This notebook teaches **model quantisation** step by step and shows you how to **quantise the Qwen2.5-0.5B model and run it locally** for fast output.

**What you will learn**
1. What quantisation is and the number formats (FP32 / FP16 / INT8 / INT4)
2. *Why* we quantise — smaller, faster, cheaper, runs offline
3. The intuition with a tiny hands-on example
4. Types of quantisation (PTQ vs QAT) and the popular tools (bitsandbytes, GPTQ, AWQ, GGUF, quanto)
5. How to pick a method for **your** hardware
6. **Hands-on:** quantise Qwen2.5-0.5B, save it locally, and run it
7. The companion scripts: `quantize_qwen.py` and `run_quantized.py`
8. Bonus: the **fastest CPU** route with GGUF / llama.cpp

> **Audience:** beginners to intermediate. Every concept is explained in plain language with runnable code.

## 1. What is quantisation?

A neural network is just a huge pile of **numbers** (called *weights*). By default each number is stored as a **32-bit floating point** value (FP32) — that is **4 bytes** per number.

**Quantisation = storing those numbers using fewer bits.**

Instead of 32 bits per weight, we use 16, 8, or even 4 bits. We trade a tiny bit of accuracy for a *huge* saving in memory and a nice speed-up.

| Format | Bits | Bytes/weight | Typical use |
|--------|------|--------------|-------------|
| FP32   | 32   | 4.0          | Training / original weights |
| FP16 / BF16 | 16 | 2.0       | GPU inference, half the size |
| INT8   | 8    | 1.0          | 4x smaller, very common |
| INT4   | 4    | 0.5          | 8x smaller, run big models on small HW |

### The memory math
Memory ≈ **number of parameters × bytes per parameter**.

A 7-billion-parameter model:
- FP32: 7B × 4 = **28 GB**  (won't fit on most laptops)
- FP16: 7B × 2 = **14 GB**
- INT8: 7B × 1 = **7 GB**
- INT4: 7B × 0.5 = **3.5 GB**  (fits on a normal laptop!)

That is the whole reason quantisation matters: it makes models that needed a data-centre GPU run on the hardware you already own.

In [ ]:
# A quick calculator: how big is a model at each precision?
def model_memory(num_params_billions, bytes_per_param):
    return num_params_billions * 1e9 * bytes_per_param / (1024**3)  # in GB

for size in [0.5, 7, 13, 70]:
    print(f"\n{size}B parameter model:")
    for name, b in [("FP32", 4), ("FP16", 2), ("INT8", 1), ("INT4", 0.5)]:
        print(f"   {name:5s}: {model_memory(size, b):7.2f} GB")

## 2. Why quantise? (the benefits)

1. **Smaller** – 4x–8x less disk and RAM. Big models fit on small machines.
2. **Faster** – moving fewer bytes from memory to the CPU/GPU is the main bottleneck in LLM inference; less data = faster tokens.
3. **Cheaper** – no need to rent a cloud GPU; run on the laptop you have.
4. **Private & offline** – the model runs **on your machine**, so your data never leaves it.
5. **Greener** – less compute = less energy.

**The cost:** a small drop in accuracy. For INT8 it's usually negligible; for INT4 it's small and often unnoticeable for everyday tasks. Good methods (GPTQ, AWQ, NF4, GGUF k-quants) keep quality high.

## 3. The intuition — quantise a few numbers by hand

Quantisation maps a range of real numbers onto a small set of integers.

For INT8 we have 256 possible values (−128..127). The recipe (symmetric quantisation):

1. Find the largest absolute value in the tensor → `absmax`.
2. `scale = absmax / 127`.
3. `q = round(x / scale)`  → store these small integers.
4. To use the weights again: `x_approx = q * scale`  ("dequantise").

Run the cell below to see the tiny rounding error this introduces.

In [ ]:
import numpy as np

# Pretend these are some model weights (FP32).
weights = np.array([0.12, -0.85, 1.73, -0.04, 0.66, -1.20], dtype=np.float32)

# --- Quantise to INT8 (symmetric) ---
absmax = np.abs(weights).max()
scale  = absmax / 127.0
q      = np.round(weights / scale).astype(np.int8)      # the stored 8-bit integers

# --- Dequantise (what the model actually uses) ---
deq = q.astype(np.float32) * scale

print("original (fp32):", weights)
print("quantised (int8):", q)
print("dequantised     :", np.round(deq, 4))
print("max error       :", np.abs(weights - deq).max())
print(f"\nStored as int8 -> 1 byte each instead of 4 bytes: {weights.nbytes} -> {q.nbytes} bytes")

## 4. Types & tools of quantisation

### Two big families
- **PTQ – Post-Training Quantisation:** take an already-trained model and shrink it. Fast, cheap, no retraining. **This is what we use here.**
- **QAT – Quantisation-Aware Training:** simulate quantisation *during* training for the best accuracy. Powerful but expensive — not needed for running models locally.

### Popular PTQ tools (and when to use them)
| Tool | Bits | Needs NVIDIA GPU? | Best for |
|------|------|-------------------|----------|
| **bitsandbytes** (NF4/INT8) | 4 / 8 | **Yes** | Quick 4-bit loading on a CUDA GPU |
| **GPTQ** (auto-gptq) | 4 | **Yes** | High-quality 4-bit on GPU |
| **AWQ** | 4 | **Yes** | Fast, high-quality 4-bit on GPU |
| **optimum-quanto** | 8 / 4 | No (works on CPU) | Pure-PyTorch, simple, cross-platform |
| **GGUF / llama.cpp** | 2–8 | No (CPU-optimised) | **Fastest on CPU**, runs everywhere |

### How to choose for YOUR machine
- **Have an NVIDIA GPU?** → bitsandbytes / GPTQ / AWQ for 4-bit on the GPU.
- **CPU-only laptop (most people)?** → **optimum-quanto** (easiest from Python) or **GGUF/llama.cpp** (fastest).

This guide uses **optimum-quanto** for the main demo because it is pure Python and runs on any machine, then shows **GGUF** as the speed champion for CPU.

Bonus - the *fastest* CPU route: GGUF + llama.cpp

`optimum-quanto` is great for learning, but for **maximum speed on a CPU**, the **GGUF** format with **llama.cpp** has hand-optimised integer kernels and is the standard for local inference (it powers Ollama, LM Studio, etc.).

The easiest path is to download an already-quantised GGUF and run it with `llama-cpp-python`:

```bash
pip install llama-cpp-python huggingface_hub
```

```python
from llama_cpp import Llama

llm = Llama.from_pretrained(
    repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
    filename="*q4_k_m.gguf",   # 4-bit, good quality/size balance
    verbose=False,
)
out = llm.create_chat_completion(
    messages=[{"role": "user", "content": "What is quantisation in one line?"}]
)
print(out["choices"][0]["message"]["content"])
```

**GGUF quant names** you'll see: `q8_0` (8-bit), `q5_k_m`, `q4_k_m` (4-bit, recommended), `q3_k_m`, `q2_k` (smallest, lower quality). The `_k_m` variants are smarter "k-quants" that keep quality high.

> Tools like **Ollama** (`ollama run qwen2.5:0.5b`) and **LM Studio** wrap llama.cpp behind a friendly UI — mention these to your class as the no-code way to run quantised models locally.